# Attention Visualization -- MiniGPT & MiniBERT (Step 14)

This notebook trains a tiny MiniGPT and MiniBERT on the toy corpora already
defined in `src/training/train_gpt.py` / `src/training/train_bert.py`, then
visualizes:

1. Per-head attention heatmaps for a chosen layer.
2. The causal-masking pattern (MiniGPT) vs bidirectional pattern (MiniBERT).
3. The training loss / perplexity curve logged during training.

Run cells top to bottom. Everything here reuses the already-verified project
modules -- no new model code is defined in this notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt

from src.config import MiniGPTConfig, MiniBERTConfig, TrainingConfig
from src.model.mini_gpt import MiniGPT
from src.model.mini_bert import MiniBERT
from src.training.train_gpt import train as train_gpt, TOY_CORPUS
from src.training.train_bert import train as train_bert, TOY_DOCUMENTS
from src.tokenizer.bpe_tokenizer import BPETokenizer

torch.manual_seed(42)


## 1. Train a tiny MiniGPT (few epochs, fast)

In [ ]:
gpt_train_config = TrainingConfig(seed=42, num_epochs=8, batch_size=4, log_every=10)
gpt_model = train_gpt(corpus=TOY_CORPUS, train_config=gpt_train_config)
gpt_model.eval()


## 2. Visualize MiniGPT's causal attention pattern

In [ ]:
tokenizer = BPETokenizer()
tokenizer.train(TOY_CORPUS, vocab_size=128)

sentence = "attention is all you need for transformers"
ids = torch.tensor([tokenizer.encode(sentence)])
tokens = [tokenizer.id_to_token.get(i, '<UNK>') for i in ids[0].tolist()]

with torch.no_grad():
    logits, attn_per_layer = gpt_model(ids)

layer_idx = 0
attn = attn_per_layer[layer_idx][0]  # (num_heads, seq_len, seq_len)
num_heads = attn.shape[0]

fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))
if num_heads == 1:
    axes = [axes]
for h in range(num_heads):
    ax = axes[h]
    im = ax.imshow(attn[h].numpy(), cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Layer {layer_idx}, Head {h}')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90, fontsize=7)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=7)
    ax.set_xlabel('key position'); ax.set_ylabel('query position')
fig.suptitle('MiniGPT: causal attention -- notice the empty upper-right triangle\n'
             '(query i has zero weight on any future key j > i)')
plt.tight_layout()
plt.savefig('minigpt_attention_heads.png', dpi=120, bbox_inches='tight')
plt.show()


## 3. Train a tiny MiniBERT (MLM + NSP)

In [ ]:
bert_train_config = TrainingConfig(seed=42, num_epochs=15, batch_size=2, log_every=5)
bert_model = train_bert(train_config=bert_train_config)
bert_model.eval()


## 4. Visualize MiniBERT's bidirectional attention pattern

In [ ]:
bert_tokenizer = BPETokenizer()
flat_corpus = [s for doc in TOY_DOCUMENTS for s in doc]
bert_tokenizer.train(flat_corpus, vocab_size=128)

sentence_a = "bert is bidirectional"
sentence_b = "it uses masked language modeling"
cls_id = bert_tokenizer.token_to_id['<BOS>']
sep_id = bert_tokenizer.token_to_id['<EOS>']
ids_a = bert_tokenizer.encode(sentence_a)
ids_b = bert_tokenizer.encode(sentence_b)
sequence = [cls_id] + ids_a + [sep_id] + ids_b + [sep_id]
segments = [0] * (len(ids_a) + 2) + [1] * (len(ids_b) + 1)

input_ids = torch.tensor([sequence])
segment_ids = torch.tensor([segments])
bert_tokens = ['[CLS]'] + [bert_tokenizer.id_to_token.get(i, '<UNK>') for i in ids_a] + \
              ['[SEP]'] + [bert_tokenizer.id_to_token.get(i, '<UNK>') for i in ids_b] + ['[SEP]']

with torch.no_grad():
    out = bert_model(input_ids, segment_ids=segment_ids)

attn = out['attention_weights'][0][0]  # layer 0, batch 0: (num_heads, seq, seq)
num_heads = attn.shape[0]

fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))
if num_heads == 1:
    axes = [axes]
for h in range(num_heads):
    ax = axes[h]
    im = ax.imshow(attn[h].numpy(), cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Layer 0, Head {h}')
    ax.set_xticks(range(len(bert_tokens))); ax.set_xticklabels(bert_tokens, rotation=90, fontsize=7)
    ax.set_yticks(range(len(bert_tokens))); ax.set_yticklabels(bert_tokens, fontsize=7)
fig.suptitle('MiniBERT: bidirectional attention -- the full square is populated,\n'
             'unlike MiniGPT\'s masked triangle, since every position may see every other position')
plt.tight_layout()
plt.savefig('minibert_attention_heads.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. Notes / limitations

- Both models are trained on tiny toy corpora for a few seconds, purely to
  produce non-trivial (non-random-looking, but not fully converged either)
  attention patterns to visualize -- they are not meant to produce
  coherent generations or strong NSP/MLM accuracy.
- Perplexity and loss curves are printed to stdout during `train_gpt` /
  `train_bert`; for a persisted, comparable curve across runs, log the
  per-step loss into a list during training and plot it here, or wire up
  a Weights & Biases run (`wandb.init(...)`, `wandb.log({...})` inside the
  training loop) per the roadmap's Module 5 tooling recommendation --
  omitted from the training scripts themselves to avoid adding a hard
  dependency on an external account for the base project to run.
- To visualize a different layer or a real sentence of your choice, change
  `layer_idx` / `sentence` above and re-run the corresponding cell.